<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

# آموزش ترنسفورمرها: توکنایزیشن و پارامترهای تولید متن (Top-k / Top-p)

> این نوت‌بوک برای اجرا روی Google Colab (حتی نسخه‌ی رایگان با CPU) طراحی شده؛ مدل خیلی سبک است.

</div>

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

## ۱. نصب و بارگذاری کتابخانه‌ها

</div>

In [ ]:
!pip install -q transformers accelerate torch

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

درباره‌ی انتخاب مدل: `Qwen2.5-0.5B-Instruct`

- کاملاً عمومی و **بدون گیت/لایسنس** است؛ یعنی برخلاف مدل‌های Gemma یا Llama نیازی به لاگین و پذیرفتن توافق‌نامه در Hugging Face ندارید.
- نسخه‌ی ۰.۵ میلیارد پارامتری خیلی سبک است و به‌راحتی روی CPU هم قابل اجراست. اگر منابع بیشتری دارید، کافی است در سلول زیر نام مدل را به `Qwen/Qwen2.5-1.5B-Instruct` تغییر دهید تا کیفیت خروجی بهتر شود.

</div>

> **نکته:** این نوت‌بوک خودش مدل را در پوشه‌ی `models` کنار خودش کش می‌کند؛ اگر قبلاً یک‌بار اجرا کرده باشید (یا از پروژه‌ی `server.py` این پوشه را دارید)، سلول بعدی هیچ دانلودی انجام نمی‌دهد و مستقیم از روی دیسک لود می‌کند.

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "google/gemma-3-1b-it"  # مدل کوچک، باز و بدون نیاز به دسترسی ویژه



In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token="hf_...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    token="hf_..."
)


device = "cuda" if torch.cuda.is_available() else "cpu"
if not torch.cuda.is_available():
    model.to(device)



config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

## ۲. توکنایزیشن روی متن فارسی

یادآوری مفهوم: **توکنایزیشن (Tokenization)** فرآیند شکستن متن به واحدهای کوچک‌تر (توکن) است که مدل با آن‌ها کار می‌کند. توکن‌ها می‌توانند کلمه، بخشی از کلمه یا حتی یک کاراکتر باشند.

یک سنجه‌ی خوب برای «بد یا خوب بودن» توکنایزیشن یک زبان، **نسبت فشرده‌سازی** است: تعداد کاراکتر تقسیم بر تعداد توکن. هرچه این عدد بزرگ‌تر باشد (نزدیک‌تر به مقداری که برای انگلیسی می‌گیریم)، توکنایزر برای آن زبان بهینه‌تر است.

</div>

In [ ]:
def show_tokenization(text, label=""):
    ids = tokenizer.encode(text)
    tokens = tokenizer.convert_ids_to_tokens(ids)
    decoded = tokenizer.decode(ids)
    n_chars = len(text)
    n_tokens = len(ids)
    ratio = n_chars / n_tokens if n_tokens else 0

    print(f"--- {label} ---")
    print("متن:", text)
    print("تعداد کاراکتر:", n_chars, "| تعداد توکن:", n_tokens, f"| نسبت فشرده‌سازی: {ratio:.2f} کاراکتر/توکن")
    print("توکن‌ها:", tokens)
    print()
    print("بازسازی‌شده پس از decode:", decoded)
    print()

fa_sentence = "امروز هوا خیلی خوب است و من به پارک رفتم."
en_sentence = "Today the weather is very nice and I went to the park."

show_tokenization(fa_sentence, "جمله فارسی")
show_tokenization(en_sentence, "همان جمله به انگلیسی")

--- جمله فارسی ---
متن: امروز هوا خیلی خوب است و من به پارک رفتم.
تعداد کاراکتر: 41 | تعداد توکن: 14 | نسبت فشرده‌سازی: 2.93 کاراکتر/توکن
توکن‌ها: ['<bos>', 'ام', 'روز', '▁هوا', '▁خیلی', '▁خوب', '▁است', '▁و', '▁من', '▁به', '▁پارک', '▁رفت', 'م', '.']

بازسازی‌شده پس از decode: <bos>امروز هوا خیلی خوب است و من به پارک رفتم.

--- همان جمله به انگلیسی ---
متن: Today the weather is very nice and I went to the park.
تعداد کاراکتر: 54 | تعداد توکن: 14 | نسبت فشرده‌سازی: 3.86 کاراکتر/توکن
توکن‌ها: ['<bos>', 'Today', '▁the', '▁weather', '▁is', '▁very', '▁nice', '▁and', '▁I', '▁went', '▁to', '▁the', '▁park', '.']

بازسازی‌شده پس از decode: <bos>Today the weather is very nice and I went to the park.



<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

معمولاً می‌بینید که نسبت فشرده‌سازی فارسی خیلی پایین‌تر از انگلیسی است و توکنایزیشنش زیاد تمیز نیست (همان‌طور که در خروجی بالا دیدیم، برای بخش زیادی از کلمات فارسی به سطح بایت خام برمی‌گردد و رشته‌های نامفهوم مثل `Ø§Ùħ` تولید می‌کند).

به همین دلیل، از این‌جا به بعد برای بخش **تولید متن** بیشتر از **پرامپت‌های ساده‌ی انگلیسی** استفاده می‌کنیم؛ چون این مدل روی انگلیسی توکن‌های تمیز و معنادار تولید می‌کند و می‌توانیم روی خودِ مفهوم پارامترهای تولید (temperature، top-k، top-p) تمرکز کنیم، بدون اینکه کیفیت ضعیف توکنایزر فارسی مزاحم شود. (بخش توکنایزیشن بالا همچنان دقیقاً نشان می‌دهد فارسی چطور شکسته می‌شود.)


</div>

In [ ]:
paragraph = (
    "Generative AI refers to a class of machine learning models that can "
    "create new content such as text, images, or audio. One of the most "
    "important components of these models is the tokenizer, which is "
    "responsible for converting text into numbers the neural network can understand."
)

show_tokenization(paragraph, "English paragraph (longer)")
# بررسی اینکه آیا رفت و برگشت متن را دقیقا حفظ می‌کند
ids = tokenizer.encode(paragraph)
roundtrip_ok = tokenizer.decode(ids, skip_special_tokens=True).strip() == paragraph.strip()
print("رفت‌وبرگشت متن بدون خطا حفظ شد؟", roundtrip_ok)

--- English paragraph (longer) ---
متن: Generative AI refers to a class of machine learning models that can create new content such as text, images, or audio. One of the most important components of these models is the tokenizer, which is responsible for converting text into numbers the neural network can understand.
تعداد کاراکتر: 278 | تعداد توکن: 53 | نسبت فشرده‌سازی: 5.25 کاراکتر/توکن
توکن‌ها: ['<bos>', 'Gener', 'ative', '▁AI', '▁refers', '▁to', '▁a', '▁class', '▁of', '▁machine', '▁learning', '▁models', '▁that', '▁can', '▁create', '▁new', '▁content', '▁such', '▁as', '▁text', ',', '▁images', ',', '▁or', '▁audio', '.', '▁One', '▁of', '▁the', '▁most', '▁important', '▁components', '▁of', '▁these', '▁models', '▁is', '▁the', '▁tokenizer', ',', '▁which', '▁is', '▁responsible', '▁for', '▁converting', '▁text', '▁into', '▁numbers', '▁the', '▁neural', '▁network', '▁can', '▁understand', '.']

بازسازی‌شده پس از decode: <bos>Generative AI refers to a class of machine learning models that can cre

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

## ۳. تولید متن پایه (Zero-Shot) با پرامپت ساده‌ی انگلیسی

چون این مدل یک مدل «Instruct» (چت‌محور) است، برای گرفتن بهترین نتیجه باید از **قالب چت (chat template)** استفاده کنیم، نه فقط متن خام. یک مثال کوتاه از مکالمه را مستقیم همین‌جا تعریف می‌کنیم (بدون نیاز به دانلود دیتاست). همان‌طور که گفته شد، از این‌جا به بعد پرامپت‌ها و مثال‌ها را ساده و انگلیسی نگه می‌داریم تا توکنایزیشن ضعیف فارسی سد راه یادگیری مفاهیم تولید متن نشود.

</div>

In [ ]:
dialogue = (
    "#Person1#: Hi, what time does tomorrow's meeting start?\n"
    "#Person2#: Hi! 10 AM, in the second-floor conference room.\n"
    "#Person1#: Okay, should I bring the sales report too?\n"
    "#Person2#: Yes please, and print a copy for the CEO if you can.\n"
    "#Person1#: Sure thing, will do."
)

print(dialogue)

#Person1#: Hi, what time does tomorrow's meeting start?
#Person2#: Hi! 10 AM, in the second-floor conference room.
#Person1#: Okay, should I bring the sales report too?
#Person2#: Yes please, and print a copy for the CEO if you can.
#Person1#: Sure thing, will do.


In [ ]:
messages = [
    {"role": "system", "content": "You are an assistant that summarizes conversations in one or two sentences."},
    {"role": "user", "content": f"Summarize the following conversation:\n\n{dialogue}"},
]

tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

"<bos><start_of_turn>user\nYou are an assistant that summarizes conversations in one or two sentences.\n\nSummarize the following conversation:\n\n#Person1#: Hi, what time does tomorrow's meeting start?\n#Person2#: Hi! 10 AM, in the second-floor conference room.\n#Person1#: Okay, should I bring the sales report too?\n#Person2#: Yes please, and print a copy for the CEO if you can.\n#Person1#: Sure thing, will do.<end_of_turn>\n<start_of_turn>model\n"

In [ ]:
a = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
inputs = tokenizer(a, return_tensors="pt").to(device)

In [ ]:
inputs["input_ids"]

tensor([[     2,      2,    105,   2364,    107,   3048,    659,    614,  16326,
            600,  72607,  23695,    528,    886,    653,   1156,  23974, 236761,
            108, 160773,    969,    506,   2269,  12309, 236787,    108, 236865,
          13285, 236770, 176581,  18428, 236764,   1144,    990,   1677,  16922,
         236789, 236751,   5395,   1502, 236881,    107, 236865,  13285, 236778,
         176581,  18428, 236888, 236743, 236770, 236771,   8151, 236764,    528,
            506,   1855, 236772,  12380,   9232,   2978, 236761,    107, 236865,
          13285, 236770, 176581,   8623, 236764,   1374,    564,   3437,    506,
           5886,   2072,   2311, 236881,    107, 236865,  13285, 236778, 176581,
           8438,   5091, 236764,    532,   1887,    496,   4865,    573,    506,
          11279,    768,    611,    740, 236761,    107, 236865,  13285, 236770,
         176581,  26145,   3210, 236764,    795,    776, 236761,    106,    107,
            105,   4368,    

In [ ]:
def generate(messages, **gen_kwargs):
    """تابع کمکی: پیام‌های چت را می‌گیرد و متن تولیدشده را برمی‌گرداند."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    default_kwargs = dict(max_new_tokens=120, pad_token_id=tokenizer.eos_token_id)
    default_kwargs.update(gen_kwargs)

    with torch.no_grad():
        output_ids = model.generate(**inputs, **default_kwargs)

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


messages = [
    {"role": "system", "content": "You are an assistant that summarizes conversations in one or two sentences."},
    {"role": "user", "content": f"Summarize the following conversation:\n\n{dialogue}"},
]

summary = generate(messages, do_sample=False)  # حالت greedy (بدون نمونه‌گیری)
print("خلاصه (greedy):")
print(summary)

خلاصه (greedy):
Person 1 asked about the start time of tomorrow’s meeting and requested to bring the sales report and a copy for the CEO. Person 2 confirmed the meeting time and provided the necessary details.


In [ ]:


messages = [
    {"role": "system", "content": "You are a Persian speaking assistant."},
    {"role": "user", "content": "پایتخت ایران کجاست؟"},
]

summary = generate(messages, do_sample=False)  # حالت greedy (بدون نمونه‌گیری)
print(summary)

سلام! پایتخت ایران، شهر تهران است. 

(Hello! The capital of Iran is Tehran.)

How can I help you with something else?


<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

**تمرین:** متن `dialogue` بالا را با یک مکالمه‌ی دیگر (مثلاً بین دو دوست درباره‌ی برنامه‌ی آخر هفته) عوض کنید — ترجیحاً همچنان به انگلیسیِ ساده — و ببینید خلاصه چطور تغییر می‌کند.

</div>

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

## ۴. پارامترهای تولید متن: `temperature`، `top_k` و `top_p`

وقتی مدل می‌خواهد کلمه‌ی بعدی را تولید کند، برای هر توکن ممکن در واژگان یک احتمال محاسبه می‌کند. اینکه از بین این احتمال‌ها چطور یکی انتخاب شود، به پارامترهای زیر بستگی دارد:

- **`do_sample`**: اگر `False` باشد، مدل همیشه محتمل‌ترین توکن را انتخاب می‌کند (**greedy decoding** — خروجی همیشه یکسان و قابل‌پیش‌بینی، ولی گاهی خسته‌کننده/تکراری). اگر `True` باشد، نمونه‌گیری تصادفی فعال می‌شود.
- **`temperature`**: هرچه عدد کوچک‌تر (مثلاً ۰.۱) باشد، مدل محتاط‌تر و نزدیک‌تر به greedy عمل می‌کند. هرچه بزرگ‌تر باشد (مثلاً ۱.۲)، خروجی متنوع‌تر و گاهی عجیب‌تر می‌شود.
- **`top_k`**: مدل را مجبور می‌کند فقط از بین *k* توکن با بالاترین احتمال انتخاب کند (مثلاً `top_k=10` یعنی فقط ۱۰ گزینه‌ی برتر در نظر گرفته می‌شود).
- **`top_p`** (یا **nucleus sampling**): به‌جای تعداد ثابت توکن، مدل کوچک‌ترین مجموعه‌ای از توکن‌ها را انتخاب می‌کند که مجموع احتمالشان به `p` برسد (مثلاً `top_p=0.9` یعنی ۹۰٪ از جرم احتمال). این روش نسبت به `top_k` انعطاف‌پذیرتر است، چون تعداد گزینه‌ها بسته به «قطعیت» مدل در هر لحظه تغییر می‌کند.

در عمل معمولاً `top_k` و `top_p` با هم استفاده می‌شوند (اول `top_k` فیلتر می‌کند، بعد `top_p` روی همان‌ها اعمال می‌شود).

</div>

In [ ]:
# چون در پارامترهای پایین می‌خواهیم اثر تصادفی‌بودن را ببینیم، seed را ثابت می‌کنیم
def generate_seeded(messages, seed=42, **gen_kwargs):
    torch.manual_seed(seed)
    return generate(messages, **gen_kwargs)

user_prompt = [
    {"role": "system", "content": "You are a creative Persian assistant."},
    {"role": "user", "content": "یک داستان کوتاه خلاقانه پیرامون باران"},
]

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

### ۴.۱ اثر `temperature`

</div>

In [ ]:
for temp in [0.1, 0.7, 1.3]:
    text = generate_seeded(
        user_prompt, seed=42, do_sample=True, temperature=temp,
        top_k=0, top_p=1.0, max_new_tokens=60,
    )
    print(f"temperature = {temp}")
    print(text)
    print("-" * 60)

temperature = 0.1
بسیار خب، بیایید یک داستان کوتاه خلاقانه پیرامون باران را بنویسم. این باران، نه فقط یک باران معمولی است، بلکه یک باران باستانی است، بارانی که در قلب یک شهر قدیمی و فراموش شده، به نام "سِ
------------------------------------------------------------
temperature = 0.7
بسیار خب، بیایید یک داستان کوتاه خلاقانه پیرامون باران را بنویسم. این باران، نه فقط یک باران، بلکه یک داستان از گذر زمان، خاطرات، و یک حس ناامیدی و امید.

---

**نام داستان: سای
------------------------------------------------------------
temperature = 1.3
## پژواک چپلا

**(جمله آغاز می‌شود با ایجاد لمس ناگهانی پرده‌ی بارانی، رنگ‌های انعلال شده عمق فضایی شعاعند و سیگنال‌هایودی بسته به تجلی کمینی و امید بخش می‌شوند.)**

دلمس «
------------------------------------------------------------


<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

### ۴.۲ اثر `top_k`

</div>

In [ ]:
for k in [1, 5, 50]:
    text = generate_seeded(
        user_prompt, seed=42, do_sample=True, temperature=0.9,
        top_k=k, top_p=1.0, max_new_tokens=60,
    )
    print(f"top_k = {k}  (top_k=1 معادل greedy است)")
    print(text)
    print("-" * 60)

top_k = 1  (top_k=1 معادل greedy است)
بسیار خب، بیایید یک داستان کوتاه خلاقانه پیرامون باران را بنویسیم. این باران، نه فقط یک باران معمولی است، بلکه یک باران باستانی است، بارانی که خاطرات را می‌نشاند و رازهای پنهان
------------------------------------------------------------
top_k = 5  (top_k=1 معادل greedy است)
بسیار خب، بیایید یک داستان کوتاه خلاقانه پیرامون باران را بنویسم. این باران یک بارانِ خاص است، یک باران که نه فقط آب است، بلکه خاطرات و آرزوهایی است.

---

**نام داستان: سای
------------------------------------------------------------
top_k = 50  (top_k=1 معادل greedy است)
بخش اول: تپش و نامرئی

سlieferتی از اسفند، در یک خانه کوچک در نزدیکیِ شهر، در میان لایه‌های تاریک و دل‌نوازِ باران می‌سوخت.  "سُفیدی" نام یک سجاد
------------------------------------------------------------


<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

### ۴.۳ اثر `top_p` (nucleus sampling)

</div>

In [ ]:
for p in [0.1, 0.5, 0.95]:
    text = generate_seeded(
        user_prompt, seed=42, do_sample=True, temperature=0.9,
        top_k=0, top_p=p, max_new_tokens=60,
    )
    print(f"top_p = {p}")
    print(text)
    print("-" * 60)

top_p = 0.1
بسیار خب، بیایید یک داستان کوتاه خلاقانه پیرامون باران را بنویسیم. این باران، نه فقط یک باران معمولی است، بلکه یک باران باستانی است، بارانی که خاطرات را می‌نشاند و رازهای پنهان
------------------------------------------------------------
top_p = 0.5
بسیار خب، بیایید یک داستان کوتاه خلاقانه پیرامون باران را بنویسیم. این باران، نه فقط یک باران معمولی است، بلکه یک باران باستانی است، بارانی که در قلب یک شهر قدیمی و فراموش شده، می‌تابد.
------------------------------------------------------------
top_p = 0.95
بخش اول: تپش و نامرئی

سlieferتی از اسفند، در یک خانه کوچک در نزدیکیِ شهر، در میان لایه‌های تاریک و سایه‌دارِ گرمای تابستان، نشسته بود.  چوبِ چوبیِ ساج
------------------------------------------------------------


<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

### ۴.۴ ترکیب `top_k` + `top_p` + `temperature`

این ترکیب رایج‌ترین تنظیم در عمل است (مثلاً همان چیزی که در بسیاری از چت‌بات‌ها استفاده می‌شود).

</div>

In [ ]:
text = generate_seeded(
    user_prompt, seed=42, do_sample=True,
    temperature=0.8, top_k=50, top_p=0.9, max_new_tokens=60,
)
print("temperature=0.8, top_k=50, top_p=0.9")
print(text)

temperature=0.8, top_k=50, top_p=0.9
بسیار خب، بیایید یک داستان کوتاه خلاقانه پیرامون باران را بنویسم. این باران، نه فقط یک باران، بلکه یک داستان است.

**نام داستان: سایه‌های آبی**

آریا به سمت خانه می‌رفت، دستش را


<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

**تمرین‌ها:**

1. مقدار `temperature` را روی ۲.۰ ببرید و ببینید خروجی چقدر بی‌معنی می‌شود.
2. `top_k=1` را با `do_sample=False` مقایسه کنید — آیا خروجی یکسان است؟ چرا؟
3. روی متغیر `user_prompt` یک سوال متفاوت بگذارید (مثلاً یک سوال دانشی به‌جای خلاقانه) و ببینید کدام تنظیمات برای آن مناسب‌تر است — برای پاسخ‌های دقیق و واقعیت‌محور معمولاً `temperature` و `top_p` پایین‌تر بهتر است.

</div>

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

## ۵. اثر `repetition_penalty`

یکی از مشکلات رایج در decoding با `temperature` پایین یا `do_sample=False`، **تکرار شدن عبارات** است. پارامتر `repetition_penalty` (بزرگ‌تر از ۱) به توکن‌هایی که قبلاً تولید شده‌اند جریمه می‌دهد تا احتمال تکرارشان کم شود.

</div>

In [ ]:
repetitive_prompt = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write five sentences about the importance of getting enough sleep."},
]

for rp in [1.0, 1.3]:
    text = generate_seeded(
        repetitive_prompt, seed=7, do_sample=False,
        repetition_penalty=rp, max_new_tokens=100,
    )
    print(f"repetition_penalty = {rp}")
    print(text)
    print("-" * 60)

repetition_penalty = 1.0
Okay, here are five sentences about the importance of getting enough sleep:

1.  Adequate sleep is crucial for physical health, as it allows your body to repair tissues, boost your immune system, and regulate hormones.
2.  Getting enough sleep significantly improves cognitive function, enhancing memory, focus, and overall mental clarity throughout the day.
3.  Insufficient sleep can lead to a range of health problems, including increased risk of chronic diseases like heart disease and diabetes.
4.
------------------------------------------------------------
repetition_penalty = 1.3
Okay, here’s five sentences highlighting the importance of sufficient sleep: 

1.  Getting adequate sleep is crucial for your physical health; it allows your body to repair tissues and strengthens its immune system.
2.  Adequate rest improves cognitive function – boosting memory, focus, and overall mental clarity throughout the day.
3.  Insufficient sleep can significantly impact moo

<div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif; line-height:1.9;">

## ۶. جمع‌بندی

- توکنایزر این مدل روی فارسی ضعیف عمل می‌کند: چون واژگانش توکن آماده‌ی کافی برای فارسی ندارد، برای بخش زیادی از متن به سطح بایت خام برمی‌گردد (همان رشته‌های نامفهوم مثل `Ø§Ùħ` که در بخش ۲ دیدیم) و نسبت فشرده‌سازی‌اش خیلی پایین‌تر از انگلیسی است. برای همین از بخش تولید متن به بعد، پرامپت‌ها را عمداً ساده و انگلیسی نگه داشتیم.
- `do_sample=False` → خروجی قطعی و تکرارپذیر (greedy).
- `temperature` شدت تصادفی‌بودن را کنترل می‌کند.
- `top_k` و `top_p` فضای انتخاب توکن بعدی را محدود می‌کنند تا از تولید توکن‌های بی‌ربط با احتمال خیلی کم جلوگیری شود.
- `repetition_penalty` از تکرار بیش‌ازحد جلوگیری می‌کند.

برای ادامه‌ی مسیر می‌توانید مدل را با `Qwen/Qwen2.5-1.5B-Instruct` یا `Qwen/Qwen2.5-3B-Instruct` جایگزین کنید (بدون تغییر در بقیه‌ی کد) تا کیفیت خروجی را با اندازه‌های مختلف مدل مقایسه کنید. برای دیدن توکنایزیشن به‌مراتب بهتر روی فارسی، مدل‌های چندزبانه‌ی متمرکزتر روی زبان‌های غیرانگلیسی (مثل خانواده‌ی Aya) گزینه‌های خوبی برای مقایسه هستند، هرچند سنگین‌ترند.

### منابع بیشتر
- https://huggingface.co/docs/transformers/main_classes/text_generation

</div>